# BirdNET-Pi Species Stats

Local offline equivalent of the BirdNET-Pi statistics dashboard (`plotly_streamlit.py`).

**Requirements:** `pip install -r ../requirements-notebook.txt`

**Usage:**
1. Set `DB_PATH` and `RECORDINGS_DIR` in the Configuration cell below.
2. Run all cells (`Kernel → Restart & Run All`).
3. Use the widgets to explore your data.

## Configuration

In [1]:
import os

# Path to your BirdNET-Pi SQLite database
DB_PATH = os.path.expanduser('/mnt/nas/sibley_backup/peterson/db/birds.db')

# Path to extracted recordings: contains By_Date/YYYY-MM-DD/Species_Name/
RECORDINGS_DIR = os.path.expanduser('/mnt/nas/sibley_backup/peterson/recordings/Extracted')

# Your location for sunrise/sunset overlay (decimal degrees)
LATITUDE  = 40.1097
LONGITUDE = -75.1962

# ---- sanity check ----
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f'Database not found: {DB_PATH}')
if not os.path.isdir(RECORDINGS_DIR):
    print(f'Warning: recordings directory not found: {RECORDINGS_DIR}')
print('Config OK')

Config OK


## Imports

In [2]:
import sqlite3
import pandas as pd
import numpy as np
from numpy import ma
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from datetime import datetime, timedelta, date
from dateutil import tz
from sklearn.preprocessing import normalize
from suntime import Sun
import ipywidgets as widgets
from IPython.display import display, Image, Audio, clear_output

pio.templates.default = 'plotly_white'

## Load Data

In [3]:
def get_connection(path):
    uri = f'file:{path}?mode=ro'
    return sqlite3.connect(uri, uri=True, check_same_thread=False)


def load_data(conn):
    df = pd.read_sql(
        'SELECT Date, Time, Sci_Name, Com_Name, Confidence, File_Name FROM detections',
        con=conn
    )
    if df.empty:
        raise ValueError('No detections found in database.')

    # Normalise Com_Name: use the most recent common name per scientific name
    # (matches the web app behaviour — language can change over time)
    latest_com_names = df.groupby('Sci_Name').tail(1)[['Sci_Name', 'Com_Name']].set_index('Sci_Name')
    df = df.rename(columns={'Com_Name': 'Directory'})
    df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
    df = df.merge(latest_com_names, how='left', on='Sci_Name').set_index('DateTime')
    return df


conn = get_connection(DB_PATH)
df_all = load_data(conn)
print(f'Loaded {len(df_all):,} detections, {df_all["Com_Name"].nunique()} species')
print(f'Date range: {df_all.index.min().date()} → {df_all.index.max().date()}')

Loaded 2,037 detections, 29 species
Date range: 2026-03-31 → 2026-04-02


## Helper Functions

In [5]:
def hms_to_dec(t):
    return t.hour + t.minute / 60 + t.second / 3600


def hms_to_str(t):
    return f'{t.hour:02d}:{t.minute:02d}'


def sunrise_sunset_scatter(date_range):
    """Return (x_dates, y_hours, hover_text) for a sunrise+sunset line trace."""
    sun = Sun(LATITUDE, LONGITUDE)
    local_tz = tz.tzlocal()
    sunrise_list, sunset_list = [], []
    rise_text, set_text = [], []
    date_labels = []

    for d in date_range:
        dt = datetime.combine(d, datetime.min.time())
        sr = sun.get_sunrise_time(dt, local_tz)
        ss = sun.get_sunset_time(dt, local_tz)/mnt/nas/sibley_backup/peterson/db
        sunrise_list.append(sr.hour + sr.minute / 60)
        sunset_list.append(ss.hour + ss.minute / 60)
        rise_text.append(f'{sr.strftime("%H:%M")} Sunrise')
        set_text.append(f'{ss.strftime("%H:%M")} Sunset')
        date_labels.append(d.strftime('%d-%m-%Y'))

    # Concatenate sunrise + None separator + sunset (single trace, two lines)
    x = date_labels + [None] + date_labels
    y = sunrise_list + [None] + sunset_list
    text = rise_text + [None] + set_text
    return x, y, text


def recording_path(row):
    """Return (wav_path, png_path) for a detections row."""
    species_dir = row['Directory'].replace(' ', '_').replace("'", '')
    base = os.path.join(RECORDINGS_DIR, 'By_Date', row['Date'], species_dir, row['File_Name'])
    return base, base + '.png'

## Widgets

In [6]:
min_date = df_all.index.min().date()
max_date = df_all.index.max().date()

w_start = widgets.DatePicker(description='Start', value=min_date, continuous_update=False)
w_end   = widgets.DatePicker(description='End',   value=max_date, continuous_update=False)
w_resample = widgets.RadioButtons(
    options=['Raw', '15 minutes', 'Hourly', 'Daily'],
    value='15 minutes',
    description='Resample:',
    layout=widgets.Layout(width='200px')
)
w_topn = widgets.IntSlider(description='Top N', min=1, max=50, value=10, continuous_update=False)
w_species = widgets.Dropdown(description='Species', options=[], value=None)
w_recording = widgets.Dropdown(description='Recording', options=[], value=None)
w_colorscale = widgets.Dropdown(
    description='Heatmap palette',
    options=px.colors.named_colorscales(),
    value='blues'
)

RESAMPLE_MAP = {'Raw': None, '15 minutes': '15min', 'Hourly': '1h', 'Daily': '1D'}

display(widgets.HBox([
    widgets.VBox([w_start, w_end, w_resample, w_topn]),
    widgets.VBox([w_species, w_recording, w_colorscale])
]))

## Charts

Run the cell below to render all charts based on the widget selections above.
Re-run after changing any widget value.

In [8]:
out_charts   = widgets.Output()
out_audio    = widgets.Output()
display(out_charts, out_audio)


def apply_date_filter(df, start, end):
    filt = (df.index >= pd.Timestamp(start)) & (df.index < pd.Timestamp(end) + timedelta(days=1))
    return df[filt]


def apply_resample(df, resample_key):
    rule = RESAMPLE_MAP[resample_key]
    if rule is None:
        return df['Com_Name']
    return df.resample(rule)['Com_Name'].aggregate('unique').explode()


def polar_trace(hourly_row, theta):
    d = pd.Series(np.zeros(24), index=range(24))
    d = (d + hourly_row).fillna(0)
    return go.Barpolar(r=d.tolist(), theta=theta, marker_color='seagreen', showlegend=False)


POLAR_LAYOUT = dict(
    radialaxis=dict(showticklabels=False),
    angularaxis=dict(
        rotation=-90, direction='clockwise',
        tickmode='array',
        tickvals=[0,15,30,45,60,75,90,105,120,135,150,165,
                  180,195,210,225,240,255,270,285,300,315,330,345],
        ticktext=['12am','1am','2am','3am','4am','5am','6am','7am','8am',
                  '9am','10am','11am','12pm','1pm','2pm','3pm','4pm','5pm',
                  '6pm','7pm','8pm','9pm','10pm','11pm'],
    )
)

theta24 = np.linspace(0.0, 360, 24, endpoint=False).tolist()


def render(change=None):
    start = w_start.value
    end   = w_end.value
    if start is None or end is None or start > end:
        return

    resample_key = w_resample.value
    top_n = w_topn.value

    df = apply_date_filter(df_all, start, end)
    if df.empty:
        with out_charts:
            clear_output()
            print('No detections in selected date range.')
        return

    df5 = apply_resample(df, resample_key)
    species_counts = df5.value_counts()
    all_species = list(df5.value_counts().index)  # sorted by freq
    hourly = pd.crosstab(df5, df5.index.hour, dropna=True, margins=True)

    # Update species dropdown without triggering another render
    current_species = w_species.value
    w_species.unobserve_all()
    w_species.value = None  # clear before updating options to avoid TraitError
    w_species.options = all_species
    w_species.value = current_species if current_species in all_species else (all_species[0] if all_species else None)
    w_species.observe(render, names='value')

    selected_species = w_species.value
    top_n = min(top_n, len(species_counts))
    top_n_species = species_counts[:top_n]

    single_day = (start == end)

    with out_charts:
        clear_output(wait=True)

        # ── Chart 1: Top-N species bar chart ─────────────────────────────────
        fig_topn = go.Figure(
            go.Bar(
                y=top_n_species.index.tolist(),
                x=top_n_species.values.tolist(),
                orientation='h',
                marker_color='seagreen'
            )
        )
        fig_topn.update_layout(
            title=f'Top {top_n} Species  {start} → {end}',
            yaxis={'categoryorder': 'total ascending'},
            margin=dict(l=0, r=0, t=40, b=0),
            height=max(300, top_n * 25)
        )
        fig_topn.show()

        if selected_species is None or selected_species not in hourly.index:
            return

        # ── Chart 2: Hourly polar chart for selected species ──────────────────
        fig_polar = go.Figure(polar_trace(hourly.loc[selected_species], theta24))
        fig_polar.update_layout(
            title=f'{selected_species} — detections by hour  {start} → {end}',
            polar=POLAR_LAYOUT,
            height=450
        )
        fig_polar.show()

        if resample_key == 'Daily':
            # ── Chart 3 (daily mode): Time-of-day heatmap ────────────────────
            df_sp = df['Com_Name'][df['Com_Name'] == selected_species].resample('15min').count()
            df_sp.index = [df_sp.index.date, df_sp.index.time]
            pivot = df_sp.unstack().fillna(0)

            fig_x   = [d.strftime('%d-%m-%Y') for d in pivot.index]
            fig_dec = [hms_to_dec(t) for t in pivot.columns]
            fig_str = [hms_to_str(t) for t in pivot.columns]
            pivot.columns = fig_dec

            n_ticks = 12
            step = max(1, len(fig_str) // n_ticks)

            x_ss, y_ss, t_ss = sunrise_sunset_scatter(pivot.index.tolist())
            fig_heat = go.Figure(data=[
                go.Heatmap(
                    x=fig_x, y=fig_dec, z=pivot.values.T.tolist(),
                    colorscale=w_colorscale.value, showscale=False
                ),
                go.Scatter(x=x_ss, y=y_ss, mode='lines',
                           hoverinfo='text', text=t_ss,
                           line_color='orange', line_width=1, name=' ')
            ])
            fig_heat.update_layout(
                title=f'{selected_species} — detections heatmap',
                yaxis=dict(
                    tickmode='array',
                    tickvals=fig_dec[::step],
                    ticktext=fig_str[::step]
                ),
                height=500
            )
            fig_heat.show()

        else:
            # ── Chart 3 (multi/single-day mode): Daily detections bar ─────────
            daily_counts = pd.crosstab(df5, df5.index.date, margins=True)
            if selected_species in daily_counts.index:
                row = daily_counts.loc[selected_species]
                fig_daily = go.Figure(
                    go.Bar(x=[str(c) for c in daily_counts.columns[:-1]],
                           y=row.iloc[:-1].tolist(),
                           marker_color='seagreen')
                )
                total = int(hourly.loc[selected_species, 'All'])
                max_conf = df[df['Com_Name'] == selected_species]['Confidence'].max()
                median_conf = df[df['Com_Name'] == selected_species]['Confidence'].median()
                fig_daily.update_layout(
                    title=(f'{selected_species} — daily detections  '
                           f'Total: {total:,}  '
                           f'Max conf: {max_conf*100:.1f}%  '
                           f'Median conf: {median_conf*100:.1f}%'),
                    height=350
                )
                fig_daily.show()

    # ── Audio / spectrogram panel ─────────────────────────────────────────────
    recordings_df = df[df['Com_Name'] == selected_species][['Date', 'Directory', 'File_Name', 'Confidence']]
    recordings_df = recordings_df.sort_index(ascending=False)

    # Update recording dropdown
    rec_options = [(f"{r.Index.date()} {r.Index.time().strftime('%H:%M')} ({r.Confidence*100:.0f}%) — {r.File_Name}", idx)
                   for idx, r in enumerate(recordings_df.itertuples())]
    w_recording.unobserve_all()
    w_recording.value = None  # clear before updating options to avoid TraitError
    w_recording.options = rec_options
    w_recording.value   = rec_options[0][1] if rec_options else None
    w_recording.observe(lambda c: show_recording(recordings_df), names='value')
    show_recording(recordings_df)


def show_recording(recordings_df):
    if w_recording.value is None or recordings_df.empty:
        return
    row = recordings_df.iloc[w_recording.value]
    wav_path, png_path = recording_path(row)
    with out_audio:
        clear_output(wait=True)
        if os.path.exists(png_path):
            display(Image(filename=png_path, width=600))
        else:
            print(f'Spectrogram not found: {png_path}')
        if os.path.exists(wav_path):
            display(Audio(filename=wav_path))
        else:
            print(f'Recording not found: {wav_path}')


# Wire up widgets
for w in (w_start, w_end, w_resample, w_topn, w_colorscale):
    w.observe(render, names='value')
w_species.observe(render, names='value')

render()

Output()

Output()

TraitError: Invalid selection: value not found